# GNSS III Lab: Measuring Tectonic Strain from GNSS Velocities

> **Colab note:** This notebook is designed to run on **Google Colab**. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amtseismo/EPS166/blob/main/notebooks/03_gnss_strain.ipynb)

## Introduction

In this lab you will use the North America-fixed GNSS velocity field from Kreemer et al. (2022) to separate translation, rigid-body rotation, and horizontal strain. You will first complete a guided example in the Pacific Northwest, then apply the same method near the northern San Andreas fault system.

Data: [Kreemer et al. velocity archive](https://doi.org/10.7910/DVN/BICMWB)  
Direct file used here: [Harvard Dataverse file 6282416](https://dataverse.harvard.edu/file.xhtml?fileId=6282416&version=1.0)

## Learning objectives

By the end, you will be able to:

- explain why velocity and strain are different quantities
- solve for a local two-dimensional velocity gradient
- calculate principal strain rates, dilatation, maximum shear, and rotation
- interpret the results in their tectonic and spatial context
- evaluate sensitivity to station geometry

## Notebook Outline:
- [Part I: Load and inspect the velocity table](#part-i-load-and-inspect-the-velocity-table)
- [Part II: Fit translation, rotation, and strain](#part-ii-fit-translation-rotation-and-strain)
- [Part III: PNW example](#part-iii-pnw-example)
- [Part IV: Northern San Andreas example](#part-iv-northern-san-andreas-example)
- [Part V: Sensitivity to station geometry](#part-v-sensitivity-to-station-geometry)
- [Part VI: Optional extension: use more than three stations](#part-vi-optional-extension-use-more-than-three-stations)
- [Synthesis](#Synthesis)
- [Summary](#Summary)

## Setup

This notebook uses NumPy, Pandas, Matplotlib, and Python's standard-library download tools. It does not require MATLAB or a mapping package.


In [ ]:
import io
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from urllib.request import urlopen

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA_URL = "https://dataverse.harvard.edu/api/access/datafile/6282416"
LOCAL_CANDIDATES = [
    Path("../data/kreemer_2022_gps_velocities.txt"),
    Path("data/kreemer_2022_gps_velocities.txt"),
    Path("kreemer_2022_gps_velocities.txt"),
]


## Part I: Load and inspect the velocity table

The code first looks for a local course copy and otherwise downloads the archived file. Keeping a local copy in the course repository is recommended so that the lab does not depend on the Dataverse server during class.


In [ ]:
def read_velocity_bytes(raw):
    """Read a text table or the first text-like member of a ZIP archive."""
    if raw[:2] == b"PK":
        with zipfile.ZipFile(io.BytesIO(raw)) as archive:
            members = [n for n in archive.namelist() if not n.endswith("/")]
            preferred = [n for n in members if n.lower().endswith((".txt", ".csv", ".tsv", ".dat"))]
            member = (preferred or members)[0]
            raw = archive.read(member)
            print(f"Reading {member} from downloaded archive")

    # First try automatic delimiter detection, then whitespace-delimited text.
    for kwargs in [
        dict(sep=None, engine="python", comment="#"),
        dict(sep=r"\s+", engine="python", comment="#"),
    ]:
        try:
            table = pd.read_csv(io.BytesIO(raw), **kwargs)
            if table.shape[1] >= 4:
                return table
        except Exception:
            pass
    raise ValueError("The downloaded file could not be parsed as a tabular velocity file.")


local_file = next((p for p in LOCAL_CANDIDATES if p.exists()), None)

if local_file is not None:
    print(f"Loading local data: {local_file}")
    raw = local_file.read_bytes()
else:
    print("No local copy found; downloading from Harvard Dataverse...")
    try:
        with urlopen(DATA_URL, timeout=60) as response:
            raw = response.read()
    except Exception as error:
        raise RuntimeError(
            "Automatic download failed. Download Dataverse file 6282416 in a browser, "
            "save it as data/kreemer_2022_gps_velocities.txt, and rerun this cell."
        ) from error

velocity_raw = read_velocity_bytes(raw)
print(f"Rows: {len(velocity_raw):,}; columns: {velocity_raw.shape[1]}")
display(velocity_raw.head())
print("Column names:", list(velocity_raw.columns))


### Standardize the column names

Archived geodetic tables use several conventions (`Ve`, `east`, `Evel`, etc.). The helper below searches for common aliases. If automatic detection fails, edit `COLUMN_MAP` after inspecting the printed column names.

The calculation requires station name, longitude, latitude, east velocity, north velocity, and their uncertainties. Velocities and uncertainties must use the same units; this notebook converts millimeters per year to meters per year internally.


In [ ]:
COLUMN_MAP = {
    # Example manual entries: "station": "site", "ve": "Evel"
    "station": None,
    "lon": None,
    "lat": None,
    "ve": None,
    "vn": None,
    "se": None,
    "sn": None,
}

ALIASES = {
    "station": ["station", "site", "sta", "name", "code", "id"],
    "lon": ["longitude", "lon", "long"],
    "lat": ["latitude", "lat"],
    "ve": ["ve", "v_e", "eastvelocity", "east_vel", "evel", "vel_e", "east"],
    "vn": ["vn", "v_n", "northvelocity", "north_vel", "nvel", "vel_n", "north"],
    "se": ["se", "sve", "sig_e", "sigma_e", "east_sigma", "east_unc", "e_unc"],
    "sn": ["sn", "svn", "sig_n", "sigma_n", "north_sigma", "north_unc", "n_unc"],
}


def canonical(text):
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


def infer_column(columns, aliases):
    normalized = {canonical(c): c for c in columns}
    for alias in aliases:
        if canonical(alias) in normalized:
            return normalized[canonical(alias)]
    # Permit units or short descriptions appended to an otherwise exact name.
    for alias in aliases:
        a = canonical(alias)
        matches = [original for key, original in normalized.items() if key.startswith(a)]
        if len(matches) == 1:
            return matches[0]
    return None


for key in COLUMN_MAP:
    if COLUMN_MAP[key] is None:
        COLUMN_MAP[key] = infer_column(velocity_raw.columns, ALIASES[key])

print("Detected column mapping:")
for key, value in COLUMN_MAP.items():
    print(f"  {key:>7s} <- {value}")

missing = [key for key, value in COLUMN_MAP.items() if value is None]
if missing:
    raise KeyError(
        f"Could not identify {missing}. Edit COLUMN_MAP using the printed archive columns."
    )

velocity = velocity_raw[[COLUMN_MAP[k] for k in COLUMN_MAP]].copy()
velocity.columns = list(COLUMN_MAP)
for c in ["lon", "lat", "ve", "vn", "se", "sn"]:
    velocity[c] = pd.to_numeric(velocity[c], errors="coerce")
velocity = velocity.dropna(subset=["lon", "lat", "ve", "vn", "se", "sn"]).copy()
velocity["station"] = velocity["station"].astype(str)

print(f"Usable horizontal velocities: {len(velocity):,}")
display(velocity.head())


### Confirm units and corrections

The Kreemer archive accompanies a North America-fixed velocity field corrected for modeled postseismic viscoelastic deformation. Confirm from the archive metadata that the selected east and north columns are the corrected velocities rather than the correction terms themselves.

The next cell assumes the velocity columns are in **mm/yr**. Change `VELOCITY_SCALE_TO_M_PER_YR` if the archive reports different units.


In [ ]:
VELOCITY_SCALE_TO_M_PER_YR = 1e-3  # mm/yr -> m/yr

print(velocity[["ve", "vn"]].describe())
print("Maximum horizontal speed:", np.hypot(velocity.ve, velocity.vn).max(), "input units/yr")


## Map the western velocity field

The arrows are velocities relative to North America. Look for coherent translation, broad turning of the vectors, and sharp spatial gradients.


In [ ]:
def add_state_outlines(ax, state_codes=("CA", "OR", "WA", "NV", "ID")):
    try:
        from bokeh.sampledata.us_states import data as states
        for code in state_codes:
            ax.plot(states[code]["lons"], states[code]["lats"], color="0.35", lw=0.8, zorder=0)
    except Exception:
        pass


fig, ax = plt.subplots(figsize=(9, 10))
add_state_outlines(ax)
speed = np.hypot(velocity.ve, velocity.vn)
q = ax.quiver(
    velocity.lon, velocity.lat, velocity.ve, velocity.vn, speed,
    angles="xy", scale_units="xy", scale=130, cmap="turbo", width=0.0022,
)
ax.set(xlim=(-126, -116), ylim=(31, 50), xlabel="Longitude", ylabel="Latitude",
       title="North America-fixed horizontal GNSS velocities")
ax.set_aspect(1 / np.cos(np.deg2rad(40)))
plt.colorbar(q, ax=ax, label="Horizontal speed (input units/yr)", shrink=0.75)
plt.show()


> **Interpret the velocity field:**
> 1. In which part of the map are velocities largest?
> 2. How does the direction change from southern California to Washington?
> 3. Give one example of two nearby regions that appear to move almost rigidly together.
> 4. Give one example of a region with a strong velocity gradient.
> 5. Why can the broad turning of the vectors not be interpreted directly as strain?

## Part II: Fit translation, rotation, and strain

For local east and north coordinates $(x,y)$, we fit

$$
v_E=t_E-\omega y+\dot\epsilon_{EE}x+\dot\epsilon_{EN}y,
$$

$$
v_N=t_N+\omega x+\dot\epsilon_{EN}x+\dot\epsilon_{NN}y.
$$

This is the same model used by the supplied GETSI MATLAB calculator, generalized here so that it can also use more than three stations.


In [ ]:
EARTH_RADIUS_M = 6_371_000.0


def local_xy(lon, lat):
    """Convert lon/lat to local east/north meters with an equirectangular projection."""
    lon = np.asarray(lon, dtype=float)
    lat = np.asarray(lat, dtype=float)
    lon0 = np.mean(lon)
    lat0 = np.mean(lat)
    x = EARTH_RADIUS_M * np.cos(np.deg2rad(lat0)) * np.deg2rad(lon - lon0)
    y = EARTH_RADIUS_M * np.deg2rad(lat - lat0)
    return x, y, lon0, lat0


def fit_horizontal_strain(stations, velocity_scale=1e-3):
    """Weighted least-squares translation, rotation, and strain estimate."""
    if len(stations) < 3:
        raise ValueError("At least three non-collinear stations are required.")

    x, y, lon0, lat0 = local_xy(stations.lon, stations.lat)
    n = len(stations)
    G = np.zeros((2*n, 6))
    d = np.zeros(2*n)
    sigma = np.zeros(2*n)

    for i in range(n):
        G[2*i] = [1, 0, -y[i], x[i], y[i], 0]
        G[2*i+1] = [0, 1, x[i], 0, x[i], y[i]]
        d[2*i:2*i+2] = [stations.iloc[i].ve, stations.iloc[i].vn]
        sigma[2*i:2*i+2] = [stations.iloc[i].se, stations.iloc[i].sn]

    d *= velocity_scale
    sigma *= velocity_scale
    sigma = np.where(sigma > 0, sigma, np.nanmedian(sigma[sigma > 0]))
    Gw = G / sigma[:, None]
    dw = d / sigma
    m, _, rank, singular_values = np.linalg.lstsq(Gw, dw, rcond=None)
    if rank < 6:
        raise ValueError("Station geometry does not independently constrain all six parameters.")

    covariance = np.linalg.inv(Gw.T @ Gw)
    model_sigma = np.sqrt(np.diag(covariance))
    predicted = G @ m
    residual = d - predicted

    strain_tensor = np.array([[m[3], m[4]], [m[4], m[5]]])
    principal, axes = np.linalg.eigh(strain_tensor)
    order = np.argsort(principal)[::-1]
    principal = principal[order]
    axes = axes[:, order]
    azimuth = (np.degrees(np.arctan2(axes[0], axes[1])) + 360) % 180

    result = {
        "centroid_lon": lon0,
        "centroid_lat": lat0,
        "translation_e_mm_yr": m[0] * 1e3,
        "translation_n_mm_yr": m[1] * 1e3,
        "rotation_nrad_yr": m[2] * 1e9,
        "rotation_direction": "counterclockwise" if m[2] > 0 else "clockwise",
        "e1_nstrain_yr": principal[0] * 1e9,
        "e2_nstrain_yr": principal[1] * 1e9,
        "e1_azimuth_deg": azimuth[0],
        "e2_azimuth_deg": azimuth[1],
        "dilatation_nstrain_yr": principal.sum() * 1e9,
        "max_shear_nstrain_yr": (principal[0] - principal[1]) * 1e9,
        "condition_number": singular_values[0] / singular_values[-1],
        "rms_residual_mm_yr": np.sqrt(np.mean(residual**2)) * 1e3,
        "strain_tensor_per_yr": strain_tensor,
        "principal_axes": axes,
        "model": m,
        "model_sigma": model_sigma,
        "predicted_m_yr": predicted.reshape(-1, 2),
        "residual_m_yr": residual.reshape(-1, 2),
    }
    return result


def result_table(result):
    keys = [
        "translation_e_mm_yr", "translation_n_mm_yr",
        "rotation_nrad_yr", "rotation_direction",
        "e1_nstrain_yr", "e1_azimuth_deg",
        "e2_nstrain_yr", "e2_azimuth_deg",
        "dilatation_nstrain_yr", "max_shear_nstrain_yr",
        "condition_number", "rms_residual_mm_yr",
    ]
    return pd.DataFrame({"value": [result[k] for k in keys]}, index=keys)


### Sanity checks

Before using real data, verify that the calculation distinguishes translation, rotation, and strain. These tests should run without an assertion error.


In [ ]:
synthetic = pd.DataFrame({
    "station": ["A", "B", "C", "D"],
    "lon": [-123.2, -122.8, -123.1, -122.7],
    "lat": [44.8, 44.8, 45.2, 45.15],
    "se": [0.1]*4,
    "sn": [0.1]*4,
})
x, y, _, _ = local_xy(synthetic.lon, synthetic.lat)

# Translation only: every station moves 10 mm/yr east and 4 mm/yr north.
synthetic["ve"] = 10.0
synthetic["vn"] = 4.0
test = fit_horizontal_strain(synthetic)
assert abs(test["e1_nstrain_yr"]) < 1e-6
assert abs(test["e2_nstrain_yr"]) < 1e-6

# Add 100 nanostrain/yr of east-west extension.
synthetic["ve"] = 10.0 + (100e-9 * x) * 1e3
test = fit_horizontal_strain(synthetic)
assert np.isclose(test["e1_nstrain_yr"], 100.0, atol=1e-6)
print("Sanity checks passed.")


## Part III: PNW example

The target points below select nearby stations automatically, avoiding dependence on particular station names. They form a broad triangle within UTM Zone 10, sampling the Cascadia forearc and the region farther inland.


In [ ]:
def nearest_unique_stations(table, targets):
    chosen = []
    used = set()
    for lon_target, lat_target in targets:
        distance2 = (
            ((table.lon - lon_target) * np.cos(np.deg2rad(lat_target)))**2
            + (table.lat - lat_target)**2
        )
        for index in distance2.sort_values().index:
            if index not in used:
                chosen.append(index)
                used.add(index)
                break
    return table.loc[chosen].copy().reset_index(drop=True)


PNW_TARGETS = [
    (-124.0, 46.2),  # coastal Washington
    (-123.1, 44.4),  # western Oregon
    (-121.6, 46.0),  # inland Washington/Oregon
]

pnw = nearest_unique_stations(velocity, PNW_TARGETS)
display(pnw)


In [ ]:
def plot_station_triangle(stations, title):
    fig, ax = plt.subplots(figsize=(8, 7))
    add_state_outlines(ax)
    polygon = pd.concat([stations, stations.iloc[[0]]])
    ax.plot(polygon.lon, polygon.lat, color="0.25", lw=1.5, zorder=1)
    ax.scatter(stations.lon, stations.lat, s=65, color="#2166ac", zorder=3)
    ax.quiver(stations.lon, stations.lat, stations.ve, stations.vn,
              angles="xy", scale_units="xy", scale=70, color="#b2182b", width=0.006)
    for row in stations.itertuples():
        ax.text(row.lon + 0.04, row.lat + 0.04, row.station, weight="bold")
    ax.set(xlabel="Longitude", ylabel="Latitude", title=title)
    ax.set_aspect(1 / np.cos(np.deg2rad(stations.lat.mean())))
    plt.show()


plot_station_triangle(pnw, "PNW station triangle and observed velocities")


### Predict before calculating

1. Does the triangle appear to translate? In what direction?
2. Do the relative velocities suggest clockwise or counterclockwise rotation?
3. In what approximate direction do you expect maximum contraction or extension?
4. Which part of your prediction is hardest to make by eye?


In [ ]:
pnw_result = fit_horizontal_strain(pnw, VELOCITY_SCALE_TO_M_PER_YR)
display(result_table(pnw_result))


In [ ]:
def plot_relative_velocities(stations, result, title):
    observed = stations[["ve", "vn"]].to_numpy()
    translation = np.array([
        result["translation_e_mm_yr"], result["translation_n_mm_yr"]
    ])
    relative = observed - translation

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax in axes:
        add_state_outlines(ax)
        polygon = pd.concat([stations, stations.iloc[[0]]])
        ax.plot(polygon.lon, polygon.lat, color="0.4", lw=1.2)
        ax.scatter(stations.lon, stations.lat, s=50, color="k", zorder=3)
        ax.set_aspect(1 / np.cos(np.deg2rad(stations.lat.mean())))
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

    axes[0].quiver(stations.lon, stations.lat, observed[:, 0], observed[:, 1],
                   angles="xy", scale_units="xy", scale=70, color="#2166ac", width=0.007)
    axes[0].set_title("Observed velocities")
    axes[1].quiver(stations.lon, stations.lat, relative[:, 0], relative[:, 1],
                   angles="xy", scale_units="xy", scale=70, color="#b2182b", width=0.007)
    axes[1].set_title("After subtracting translation")
    fig.suptitle(title, weight="bold")
    plt.tight_layout()
    plt.show()


plot_relative_velocities(pnw, pnw_result, "Translation removal reveals rotation and strain")


> ## **Interpret the PNW result:**
> 1. Report the rotation rate and direction.
> 2. Report $\dot\epsilon_1$ and $\dot\epsilon_2$, including signs and units.
> 3. Is the triangle undergoing net areal expansion or contraction?
> 4. Which principal axis is most relevant to Cascadia margin-normal shortening?
> 5. Does this triangle isolate Cascadia locking, or does it mix multiple tectonic domains?
> 6. Because three stations exactly determine six parameters, what does an RMS residual near zero mean—and what does it **not** mean?

## Part IV: Northern San Andreas example

Now repeat the analysis near the northern San Andreas system. The default targets form a triangle between the coast near the Mendocino region, the northern Coast Ranges, and the Point Reyes–San Francisco region. You may adjust them to test a different portion of the fault system.


In [ ]:
NORCAL_TARGETS = [
    (-124.05, 40.15),
    (-122.75, 39.55),
    (-122.95, 38.15),
]

norcal = nearest_unique_stations(velocity, NORCAL_TARGETS)
display(norcal)
plot_station_triangle(norcal, "Northern California station triangle")


### Your analysis

Before running the calculation, sketch or describe your expected deformation. Consider the right-lateral San Andreas system, distributed Coast Range deformation, and proximity to the Mendocino Triple Junction.


In [ ]:
norcal_result = fit_horizontal_strain(norcal, VELOCITY_SCALE_TO_M_PER_YR)
display(result_table(norcal_result))
plot_relative_velocities(norcal, norcal_result, "Northern California velocity decomposition")


> ## **Interpret northern California:**
> 1. Is rotation clockwise or counterclockwise? Is that consistent with the map?
> 2. What are the orientations of maximum extension and maximum contraction?
> 3. Would a right-lateral fault produce principal axes parallel and perpendicular to the fault, or oblique to it?
> 4. Does the result appear dominated by strike-slip shear, extension, contraction, or a combination?
> 5. Which known tectonic structures fall inside the triangle?
> 6. What additional information would you want before assigning the strain to one fault?

## Part V: Sensitivity to station geometry

A three-station strain estimate is not unique to a named region; it is specific to the selected triangle. Move one target by roughly 0.5–1° or replace one selected station with another nearby station. Then recompute the result.


In [ ]:
NORCAL_ALTERNATIVE_TARGETS = [
    (-123.75, 40.00),  # modified from the first target
    (-122.75, 39.55),
    (-122.95, 38.15),
]

norcal_alternative = nearest_unique_stations(velocity, NORCAL_ALTERNATIVE_TARGETS)
alternative_result = fit_horizontal_strain(norcal_alternative, VELOCITY_SCALE_TO_M_PER_YR)

comparison = pd.DataFrame({
    "original": result_table(norcal_result)["value"],
    "alternative": result_table(alternative_result)["value"],
})
display(norcal_alternative)
display(comparison)
plot_station_triangle(norcal_alternative, "Alternative northern California triangle")


> ## **Evaluate sensitivity:**
> 1. Which result changed most: translation, rotation, principal strain magnitude, or principal-axis orientation?
> 2. Did the sign of either principal strain change?
> 3. Which tectonic boundary or velocity gradient was added to or removed from the triangle?
> 4. Which interpretation is robust to the station change?
> 5. Write one sentence explaining why a published strain-rate map should not be interpreted without knowing its spatial smoothing or station geometry.

## Part VI: Optional extension: use more than three stations

The function accepts additional stations and solves an overdetermined weighted least-squares problem. Select all stations in a compact bounding box and compare the result with a triangle spanning the same region.


In [ ]:
# Modify these bounds if time permits.
regional = velocity[
    velocity.lon.between(-124.2, -122.5)
    & velocity.lat.between(38.2, 40.3)
].copy()

print(f"Regional solution uses {len(regional)} stations")
if len(regional) >= 4:
    regional_result = fit_horizontal_strain(regional, VELOCITY_SCALE_TO_M_PER_YR)
    display(result_table(regional_result))
else:
    print("Fewer than four stations found; expand the bounding box.")


---
## Synthesis

> **In a short paragraph, compare the PNW and northern California results. Your response should:**
> 1. distinguish rigid rotation from strain;
> 2. compare the signs and orientations of the principal strain rates;
> 3. connect each result to the regional plate boundary;
> 4. describe at least one limitation caused by station geometry or spatial scale.

**References**
- Kreemer, C., Hammond, W. C., and Blewitt, G. (2022), *Crustal Strain Rates in the Western United States and Their Relationship with Earthquake Rates*, **Seismological Research Letters**, 93, 2990–3006.
- Kreemer et al. velocity dataset: https://doi.org/10.7910/DVN/BICMWB
- GETSI/EarthScope, *Infinitesimal Strain Analysis Using GPS Data*.
- Savage et al. (2001), method underlying the GETSI `calcstrain.m` implementation.

## Summary
